# Traffic Sign Classification with Convolutional Neural Networks

A PyTorch deep learning project comparing five CNN architectures on a 4-class road sign detection task.

**Dataset:** [Kaggle Road Sign Detection](https://www.kaggle.com/datasets/andrewmvd/road-sign-detection)  
**Classes:** Traffic Light · Stop · Speed Limit · Crosswalk  
**Models:** Baseline CNN → Max Pooling → Batch Normalization → Dropout → ResNet

---

All model implementations live in `models.py`. This notebook covers data loading, training, hyperparameter experiments, and a final model comparison.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import matplotlib.pyplot as plt
from PIL import Image

import os
import pickle
import sys

I used Vectorized operations throughout, because they leverage PyTorch's parallelism efficiently and produce cleaner code.

## Setup

This project was run on **Google Colab Pro** (A100 GPU) to handle the computational demands of training multiple CNN architectures, including ResNet, across 40 epochs. Any Hardware accelerator should work, but A100 GPU will ensure speed.


Run the cells below to install dependencies, download the dataset, and verify GPU availability.

Click Choose Files → navigate to your kaggle.json on your computer → select it → the cell will finish running and download the dataset.

If you don't have your kaggle.json yet:

1. Go to kaggle.com
2. Click your profile icon → Settings
3. Scroll to API section
4. Click Create New Token by Clicking "Create Legacy API Key"
5. It downloads kaggle.json to your computer automatically

In [ ]:
# Install the Kaggle API and download the Road Sign Detection dataset
!pip install kaggle -q

# Upload your kaggle.json API token when prompted, or place it at ~/.kaggle/kaggle.json
from google.colab import files
files.upload()  # upload kaggle.json here

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d andrewmvd/road-sign-detection -p data/ --unzip
print("Dataset downloaded.")

In [ ]:
import sys, os

# If running in Colab, clone the repo so models.py is available
if "google.colab" in sys.modules:
    repo = "traffic-sign-cnn-pytorch"
    if not os.path.exists(repo):
        os.system(f"git clone https://github.com/Walekazam/{repo}")
    sys.path.insert(0, repo)
else:
    # Running locally after cloning the repo — models.py is already here
    sys.path.insert(0, ".")

import models

# Auto-reload models whenever models.py is edited
%load_ext autoreload
%aimport models
%autoreload 1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Dataset

The [Road Sign Detection dataset](https://www.kaggle.com/datasets/andrewmvd/road-sign-detection) contains annotated images of four traffic sign classes. Raw images are loaded from the downloaded `data/` folder, resized to 64×64, normalized, and serialized into `.pkl` splits for efficient loading during training.

**Classes:** `trafficlight`, `stop`, `speedlimit`, `crosswalk`

In [ ]:
classes = ['trafficlight', 'stop', 'speedlimit', 'crosswalk']

### 1.1 Preprocessing

The cell below builds train/val/test `.pkl` splits from the raw Kaggle images. Run this once — it saves `train_data.pkl`, `val_data.pkl`, and `test_data.pkl` to `data/`. If you already have the `.pkl` files, skip to section 1.2.

In [ ]:
import os, pickle, random
from PIL import Image
import xml.etree.ElementTree as ET
from torchvision import transforms

data_mean = [0.4910, 0.4884, 0.5115]
data_std  = [0.2489, 0.2475, 0.2343]

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x[:3, :, :]),
    transforms.Normalize(data_mean, data_std),
])

class_to_idx = {'trafficlight': 0, 'stop': 1, 'speedlimit': 2, 'crosswalk': 3}

images_dir = 'data/images'
annots_dir = 'data/annotations'

samples = []
for fname in os.listdir(annots_dir):
    if not fname.endswith('.xml'):
        continue
    tree = ET.parse(os.path.join(annots_dir, fname))
    root = tree.getroot()
    img_file = root.find('filename').text
    img_path = os.path.join(images_dir, img_file)
    if not os.path.exists(img_path):
        continue
    for obj in root.findall('object'):
        label_str = obj.find('name').text.lower().replace(' ', '')
        if label_str not in class_to_idx:
            continue
        label = class_to_idx[label_str]
        img = Image.open(img_path).convert('RGB')
        tensor = transform(img)
        samples.append((tensor, label))

random.seed(42)
random.shuffle(samples)

n = len(samples)
train_end = int(0.7 * n)
val_end   = int(0.85 * n)

splits = {
    'train': samples[:train_end],
    'val':   samples[train_end:val_end],
    'test':  samples[val_end:]
}

for split, data in splits.items():
    path = f'data/{split}_data.pkl'
    with open(path, 'wb') as f:
        pickle.dump(data, f)
    print(f"Saved {len(data)} samples → {path}")

### 1.2 Loading the Preprocessed Splits

The `PickleDataset` class below wraps the `.pkl` splits into a standard PyTorch `Dataset`.

In [ ]:
class PickleDataset(Dataset):
    def __init__(self, pkl_file, test_mode=False):
        """
        Args:
            pkl_file (str):   Path to the .pkl split file.
            test_mode (bool): If True, returns images only (no labels).
        """
        with open(pkl_file, 'rb') as f:
            self.data = pickle.load(f)
        self.test_mode = test_mode

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image, label = self.data[idx]
        return image if self.test_mode else (image, label)


def load_dataset(split, data_dir='data', test_mode=False):
    """Load a train/val/test split from a .pkl file in data_dir."""
    pkl_file = os.path.join(data_dir, f"{split}_data.pkl")
    dataset = PickleDataset(pkl_file, test_mode=test_mode)
    print(f"Loaded {len(dataset)} samples  ← {pkl_file}")
    return dataset

Load train, validation, and test splits into DataLoaders with batch size 32.

In [ ]:
train_dataset = load_dataset("train")
val_dataset   = load_dataset("val")
test_dataset  = load_dataset("test", test_mode=True)

BATCH_SIZE = 32
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

The following normalization and resize transforms were applied during preprocessing:

```python
data_mean = [0.4910, 0.4884, 0.5115]
data_std  = [0.2489, 0.2475, 0.2343]

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x[:3, :, :]),
    transforms.Normalize(data_mean, data_std),
])
```

Visualize a sample image and verify DataLoader output shapes.

In [ ]:
image, label = train_dataset[4]
to_pil = transforms.ToPILImage()
plt.imshow(to_pil(image))
plt.title(f"Label: {classes[label]}")
plt.axis('off')
plt.show()

In [ ]:
for batch_idx, (data, target) in enumerate(train_dataloader):
    print(f"Batch {batch_idx}:")
    print(f"  Data   — shape: {data.shape},   dtype: {data.dtype}")
    print(f"  Target — shape: {target.shape}, dtype: {target.dtype}")
    if batch_idx == 2:
        break

## 2. Training Pipeline

The `train()` function in `models.py` implements a standard supervised training loop:

1. Zero gradients
2. Forward pass
3. Compute cross-entropy loss
4. Backward pass
5. Optimizer step

Train and validation losses are tracked per epoch and returned as arrays for plotting.

In [ ]:
criterion = nn.CrossEntropyLoss()

## 3. Baseline CNN

The baseline model uses three strided convolutional layers (no pooling) followed by two fully connected layers.

**Architecture:**
- `conv1`: 3 → 4 channels, kernel 3, stride 2, padding 1
- `conv2`: 4 → 16 channels, kernel 3, stride 2, padding 1
- `conv3`: 16 → 32 channels, kernel 3, stride 2, padding 1
- `fc1`: 2048 → 1024
- `fc2`: 1024 → 4 classes

ReLU activations throughout. See `models.py → ConvNet`.

Verify output shape is `(batch, 4)`.

In [ ]:
def test_convnet():
    net = models.ConvNet()
    out = net(torch.randn(1, 3, 64, 64))
    assert out.shape == torch.Size([1, 4]), f"Expected (1,4), got {out.shape}"
    print("ConvNet shape test passed.")

test_convnet()

### 3.1 Learning Rate Sweep

Train the baseline CNN across multiple learning rates to study convergence sensitivity.

In [ ]:
train_loss_results = {}
val_loss_results   = {}

for lr in [1e-2, 1e-3, 1e-4, 5e-4]:
    print(f"{'─'*40}\nLearning rate: {lr}")
    model = models.ConvNet().to(device)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    t_loss, v_loss = models.train(model, train_dataloader, val_dataloader, criterion, optimizer, epochs=20, device=device)
    train_loss_results[lr] = t_loss
    val_loss_results[lr]   = v_loss

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for lr, loss in train_loss_results.items():
    ax1.plot(loss, label=f"lr={lr}")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Training Loss vs Epoch"); ax1.legend()

for lr, loss in val_loss_results.items():
    ax2.plot(loss, label=f"lr={lr}")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
ax2.set_title("Validation Loss vs Epoch"); ax2.legend()

plt.tight_layout()
plt.show()

**Observation:** Lower learning rates (1e-3, 1e-4) converge more stably with lower final validation loss. The highest rate (1e-2) shows instability or early divergence, confirming that the baseline architecture is sensitive to learning rate and benefits from careful tuning.

## 4. Adding Max Pooling

`ConvNetMaxPooling` extends the baseline by adding 2×2 max pooling (stride 2) after each conv block's ReLU, reducing spatial dimensions more aggressively and producing a more compact feature representation before the FC layers.

See `models.py → ConvNetMaxPooling`.

In [ ]:
def test_convnet_maxpool():
    net = models.ConvNetMaxPooling()
    out = net(torch.randn(1, 3, 64, 64))
    assert out.shape == torch.Size([1, 4]), f"Expected (1,4), got {out.shape}"
    print("ConvNetMaxPooling shape test passed.")

test_convnet_maxpool()

## 5. Custom Batch Normalization

`BatchNormalization` is implemented from scratch using PyTorch primitives — no `nn.BatchNorm2d` used internally.

During training, batch statistics are computed over `(B, H, W)` per channel and used to normalize activations. Running statistics are maintained via exponential moving average for inference. Learnable scale (γ) and shift (β) parameters are applied per channel.

See `models.py → BatchNormalization` and `ConvNetBN`.

In [ ]:
def test_batchnorm():
    torch.manual_seed(42)
    np.random.seed(42)

    custom_bn = models.BatchNormalization(num_features=3)
    torch_bn  = nn.BatchNorm2d(num_features=3)
    custom_bn.train(); torch_bn.train()

    for _ in range(50):
        x = torch.tensor(np.random.randn(10, 3, 28, 28), dtype=torch.float32)
        out_custom = custom_bn(x).detach().numpy()
        out_torch  = torch_bn(x).detach().numpy()
        assert np.allclose(out_custom, out_torch, atol=1e-3)

    # Inference mode check
    x = torch.tensor(np.random.randn(10, 3, 28, 28), dtype=torch.float32)
    custom_bn.eval(); torch_bn.eval()
    out_custom = custom_bn(x).detach().numpy()
    out_torch  = torch_bn(x).detach().numpy()
    print(f"Max inference diff vs nn.BatchNorm2d: {np.max(np.abs(out_custom - out_torch)):.6f}")
    assert np.allclose(out_custom, out_torch, atol=1e-3)
    print("BatchNormalization test passed.")

test_batchnorm()

## 6. Custom Dropout

`CustomDropout` implements inverted dropout from scratch: at training time, activations are randomly zeroed with probability `p` and surviving activations are scaled by `1 / (1 - p)`. At inference time the layer is a pass-through identity.

See `models.py → CustomDropout` and `ConvNetDropout`.

In [ ]:
def test_dropout():
    torch.manual_seed(42)

    class LinearDropout(nn.Module):
        def __init__(self, num_features, p=0.1):
            super().__init__()
            self.linear  = nn.Linear(num_features, 1)
            self.linear.weight.data.fill_(1)
            self.dropout = models.CustomDropout(p)
        def forward(self, x):
            return self.linear(self.dropout(x))

    model = LinearDropout(512)
    x = torch.ones((1000, 512))

    # Inference: deterministic
    model.eval()
    out = model(x)
    assert torch.allclose(out.std(), torch.zeros(1), atol=1e-5), "Eval mode should be deterministic"

    # Training: stochastic
    model.train()
    outputs = torch.cat([model(x) for _ in range(50)])
    assert outputs.std() > 1, "Train mode should introduce variance"
    print("CustomDropout test passed.")

test_dropout()

## 7. ResNet

### 7.1 Residual Block

`ResidualBlock` is the core building block. The skip connection adds the input directly to the main branch output, enabling gradients to flow unimpeded during backpropagation.

**Main branch:** conv1 (3×3) → BN → ReLU → conv2 (3×3) → BN → conv3 (3×3)  
**Skip branch:** 1×1 projection conv when channel dimensions differ, else Identity  
**Output:** ReLU(main + skip)

See `models.py → ResidualBlock`.

In [ ]:
def test_residual_block():
    blk = models.ResidualBlock(in_channel=9, interm_channel=6, out_channel=3)
    X   = torch.randn(4, 9, 6, 6)
    assert blk.conv1(X).shape          == torch.Size([4, 6, 6, 6])
    assert blk.conv2(blk.conv1(X)).shape == torch.Size([4, 3, 6, 6])
    assert blk(X).shape                == torch.Size([4, 3, 6, 6])
    print("ResidualBlock test passed.")

test_residual_block()

### 7.2 Full ResNet

`ResNet` stacks residual blocks into two configurable block layers with a 7×7 stem at the front and adaptive average pooling into a linear classifier at the back.

**Architecture:**  
Stem (7×7 conv → BN → ReLU → MaxPool) → block_layer1 → block_layer2 → AdaptiveAvgPool → Linear(num_classes)

See `models.py → ResNet`.

In [ ]:
def test_resnet():
    net = models.ResNet(num_blocks=2, layer1_channel=64, layer2_channel=128, out_channel=256)
    X   = torch.randn(1, 1, 96, 96)
    l1_out = net.layer1(net.first(X))
    l2_out = net.layer2(l1_out)
    assert l1_out.shape[1] == 128
    assert l2_out.shape[1] == 256
    assert net(X).shape    == torch.Size([1, 4])
    print("ResNet test passed.")

test_resnet()

## 8. Model Comparison

All five models are trained with identical settings (SGD, lr=0.001, momentum=0.9, 40 epochs) to compare convergence behavior and generalization.

In [ ]:
epochs = 40

# ── Baseline CNN ──────────────────────────────────────────────────────────────
model_conv = models.ConvNet().to(device)
optimizer  = optim.SGD(model_conv.parameters(), lr=0.001, momentum=0.9)
convnet_train_loss, convnet_val_loss = models.train(
    model_conv, train_dataloader, val_dataloader, criterion, optimizer, epochs, device)

In [ ]:
# ── Max Pooling ───────────────────────────────────────────────────────────────
model_pool = models.ConvNetMaxPooling().to(device)
optimizer  = optim.SGD(model_pool.parameters(), lr=0.001, momentum=0.9)
convnet_max_pooling_train_loss, convnet_max_pooling_val_loss = models.train(
    model_pool, train_dataloader, val_dataloader, criterion, optimizer, epochs, device)

In [ ]:
# ── Batch Normalization ───────────────────────────────────────────────────────
model_bn  = models.ConvNetBN().to(device)
optimizer = optim.SGD(model_bn.parameters(), lr=0.001, momentum=0.9)
convnet_bn_train_loss, convnet_bn_val_loss = models.train(
    model_bn, train_dataloader, val_dataloader, criterion, optimizer, epochs, device)

In [ ]:
# ── Dropout ───────────────────────────────────────────────────────────────────
model_dropout = models.ConvNetDropout().to(device)
optimizer     = optim.SGD(model_dropout.parameters(), lr=0.001, momentum=0.9)
convnet_dropout_train_loss, convnet_dropout_val_loss = models.train(
    model_dropout, train_dataloader, val_dataloader, criterion, optimizer, epochs, device)

In [ ]:
# ── ResNet ────────────────────────────────────────────────────────────────────
model_res = models.ResNet(num_blocks=2, layer1_channel=12, layer2_channel=64, out_channel=128).to(device)
optimizer = optim.SGD(model_res.parameters(), lr=0.001, momentum=0.9)
resnet_train_loss, resnet_val_loss = models.train(
    model_res, train_dataloader, val_dataloader, criterion, optimizer, epochs, device)

Plot training and validation loss curves for all five architectures.

In [ ]:
x = list(range(epochs))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for label, loss, color in [
    ('ConvNet',          convnet_train_loss,           'red'),
    ('MaxPooling',       convnet_max_pooling_train_loss,'green'),
    ('BatchNorm',        convnet_bn_train_loss,         'gold'),
    ('Dropout',          convnet_dropout_train_loss,    'orange'),
    ('ResNet',           resnet_train_loss,             'steelblue'),
]:
    ax1.plot(x, loss, color=color, label=label)

ax1.set_title("Training Loss"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend()

for label, loss, color in [
    ('ConvNet',          convnet_val_loss,              'red'),
    ('MaxPooling',       convnet_max_pooling_val_loss,  'green'),
    ('BatchNorm',        convnet_bn_val_loss,           'gold'),
    ('Dropout',          convnet_dropout_val_loss,      'orange'),
    ('ResNet',           resnet_val_loss,               'steelblue'),
]:
    ax2.plot(x, loss, color=color, label=label)

ax2.set_title("Validation Loss"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss"); ax2.legend()

plt.tight_layout()
plt.show()

**Results Summary:**

- **Baseline CNN** establishes the performance floor with reasonable but slower convergence
- **MaxPooling** improves generalization by reducing spatial dimensions, lowering validation loss relative to baseline
- **BatchNorm** produces the smoothest loss curves and fastest convergence — normalization stabilizes the gradient signal across layers
- **Dropout** shows slightly higher training loss (expected from regularization) but competitive validation performance
- **ResNet** achieves the best overall validation loss — skip connections enable more effective gradient flow and richer feature learning

Save ResNet predictions on the held-out test set.

In [ ]:
model_res.eval()
predictions = []

with torch.no_grad():
    for data in test_dataloader:
        data   = data.to(device)
        output = model_res(data)
        preds  = output.argmax(dim=1)
        predictions.extend(preds.cpu().numpy().tolist())

os.makedirs("results", exist_ok=True)
pred_df = pd.DataFrame(predictions, columns=["predicted_class"])
pred_df["class_name"] = pred_df["predicted_class"].map(dict(enumerate(classes)))
pred_df.to_csv("results/resnet_predictions.csv", index=False)
print(f"Saved {len(pred_df)} predictions → results/resnet_predictions.csv")
pred_df["class_name"].value_counts()

---
*Model implementations are in `models.py`. See `README.md` for full setup and reproduction instructions.*